# 🧑‍💻 Lab 3.2 — Pattern 1: Plain Python Function (Local Agent)

## Learning Objectives
In this notebook, you will learn:
1. **Local Agent Pattern** — Wire a locally-defined function into `mlflow.genai.evaluate()` — the fastest dev loop
2. **`@mlflow.trace`** — Decorate the agent so each invocation produces a trace linked to the eval run
3. **Signature Contract** — Use parameter names that match the dataset's `inputs.<key>` keys
4. **Built-in Judges** — Score with `Correctness` and `RelevanceToQuery` from `mlflow.genai.scorers`
5. **Versioned Runs** — Set `model_id` so the run is comparable across iterations

## Prerequisites
- Lab 1.3 — workspace setup complete
- Lab 2.2 — `tutorial_eval_v1` dataset exists in `genai_eval_tutorial.module_01`
- Foundation Model API access for `databricks-claude-sonnet-4`


In [ ]:
# ============================================================================
# 📦 INSTALL PACKAGES
# ============================================================================

%pip install --quiet "mlflow[databricks]>=3.1" databricks-openai

dbutils.library.restartPython()


---
## Step 1 — Configure Namespace and Experiment

We reuse the catalog, schema, and experiment from Modules 1 and 2. The eval results land in this experiment alongside the traces produced by `@mlflow.trace`.


In [ ]:
# ============================================================================
# 🗂️ STEP 1 - CONFIGURE NAMESPACE AND EXPERIMENT
# ============================================================================

import mlflow

CATALOG = "genai_eval_tutorial"
SCHEMA  = "module_01"
DATASET_FQN = f"{CATALOG}.{SCHEMA}.tutorial_eval_v1"

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)
EXPERIMENT_PATH = f"/Users/{USER_EMAIL}/genai-eval-tutorial"
mlflow.set_experiment(EXPERIMENT_PATH)

print(f"Experiment: {EXPERIMENT_PATH}")
print(f"Eval data:  {DATASET_FQN}")


---
## Step 2 — Load the Eval Dataset

`mlflow.genai.datasets.get_dataset(...)` returns the dataset object built in Lab 2.2. We pass `eval_dataset` directly into `evaluate()` later — MLflow understands the schema natively.


In [ ]:
# ============================================================================
# 📚 STEP 2 - LOAD THE EVAL DATASET
# ============================================================================

import mlflow.genai.datasets

eval_dataset = mlflow.genai.datasets.get_dataset(name=DATASET_FQN)
print(f"Loaded {eval_dataset.to_df().count()} rows from {eval_dataset.name}")
display(eval_dataset.to_df().limit(3))


---
## Step 3 — Build the Local Agent

Three things must be true for `evaluate()` to call this function correctly:
1. **Decorated with `@mlflow.trace`** — every invocation produces a trace, which scorers can inspect for retrieval/tool-call metadata.
2. **Parameter name matches the dataset key.** The dataset has `inputs.question`, so the function signature must be `def my_agent(question: str)`. MLflow passes each row's inputs as keyword arguments.
3. **Returns a string (or dict).** A plain string becomes the `response` column in the results.


In [ ]:
# ============================================================================
# ▶️ STEP 3 - BUILD THE LOCAL AGENT
# ============================================================================

from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()

@mlflow.trace
def my_agent(question: str) -> str:
    resp = client.chat.completions.create(
        model="databricks-claude-sonnet-4",
        messages=[
            {"role": "system", "content": "You are a Databricks expert. Answer concisely in 2-3 sentences."},
            {"role": "user",   "content": question},
        ],
    )
    return resp.choices[0].message.content

# Quick smoke test
print(my_agent("What is Delta Live Tables?"))


---
## Step 4 — Run `mlflow.genai.evaluate`

We score with two built-in judges:
- **`Correctness`** — checks the response covers the dataset's `expected_facts`.
- **`RelevanceToQuery`** — checks the response actually answers the question (catches off-topic verbosity).

Set `model_id` so eval results are versioned — re-running with a new prompt/model is one parameter change.


In [ ]:
# ============================================================================
# 🧮 STEP 4 - RUN `MLFLOW.GENAI.EVALUATE`
# ============================================================================

from mlflow.genai.scorers import Correctness, RelevanceToQuery

results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=my_agent,
    scorers=[Correctness(), RelevanceToQuery()],
    model_id="models:/my-agent/1",
)

print("Evaluation complete.")


---
## Step 5 — Inspect Results

Each row in `eval_results` carries:
- the original `inputs` and `expectations`
- the agent's `response`
- one column per scorer with the verdict and rationale
- a `trace_id` linking back to the trace produced during evaluation


In [ ]:
# ============================================================================
# 🔍 STEP 5 - INSPECT RESULTS
# ============================================================================

display(results.tables["eval_results"])


In [ ]:
# ============================================================================
# ▶️ AGGREGATE SCORER PASS RATE
# ============================================================================

agg = results.tables["eval_results"]
display(agg.selectExpr(
    "AVG(CASE WHEN `correctness/v1/value` = 'yes' THEN 1.0 ELSE 0.0 END) AS correctness_pass_rate",
    "AVG(CASE WHEN `relevance_to_query/v1/value` = 'yes' THEN 1.0 ELSE 0.0 END) AS relevance_pass_rate",
    "COUNT(*) AS total_rows",
))


---
## Step 6 — Verify Traces in the MLflow UI

Open the experiment in the MLflow UI and switch to the **Evaluations** tab. You should see:
- One eval run grouped by `model_id`
- Each row click-through opens the trace from `@mlflow.trace`
- Scorer verdicts inline next to each response

You can also fetch the linked traces programmatically:


In [ ]:
# ============================================================================
# 🔭 STEP 6 - VERIFY TRACES IN THE MLFLOW UI
# ============================================================================

traces = mlflow.search_traces(
    experiment_names=[EXPERIMENT_PATH],
    max_results=5,
    order_by=["attributes.timestamp_ms DESC"],
)
display(traces)


---
## Lab Complete

| Check | Status |
| --- | --- |
| Local agent decorated with `@mlflow.trace` | ✅ |
| `predict_fn` parameter name matches `inputs.question` | ✅ |
| `evaluate()` ran with Correctness + RelevanceToQuery | ✅ |
| Each row produced a linked trace in the UI | ✅ |
| `model_id` set so the run is versioned | ✅ |

Next: **Lab 3.3** — switch from a local function to a deployed Model Serving endpoint with no code changes to the endpoint itself.


---
## 📝 Summary

In this notebook, we covered:

### 1. Pattern 1: Local Function
- **Best for active development.** No deployment, no versioning overhead — change the code and re-run.
- MLflow passes each row's `inputs` dict as keyword arguments — parameter names matter.

### 2. Tracing Contract
- **`@mlflow.trace` is non-negotiable.** Without it, scorers that inspect retrieval / tool calls have nothing to read.
- Each evaluated row carries a `trace_id` linking back to the trace produced during evaluation.

### 3. Versioning with `model_id`
- Setting `model_id="models:/my-agent/1"` groups runs in the MLflow UI so comparisons across iterations are one click away.

### Next Steps
- **Lab 3.3** — switch to a deployed Model Serving endpoint with one line: `mlflow.genai.to_predict_fn(...)`.
